<a href="https://colab.research.google.com/github/WMaia9/Mori_RAG/blob/main/notebook/1_Geracao_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ✅ Criação dos Embedding

In [ ]:
!pip install faiss-cpu sentence-transformers pandas
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === IMPORTAÇÕES ===
import pandas as pd
import numpy as np
import faiss
import pickle
import gc
import torch
import re
from sentence_transformers import SentenceTransformer
import os

# === LIMPAR MEMÓRIA ===
gc.collect()
torch.cuda.empty_cache()

# === 1. CARREGAR DADOS ===
# ATENÇÃO: Verifique se o caminho do seu arquivo Excel está correto.
caminho_xlsx = "/content/drive/MyDrive/MORI/Base_de_ODAS_1606.xlsx"
df_odas = pd.read_excel(caminho_xlsx)

# === 2. PRÉ-FILTRAGEM ===
df_odas = df_odas.dropna(subset=["Resumo completo", "Temas", "Dimensões", "Tipo"])

# === 3. INTERPRETAR DURAÇÃO ===
def interpretar_duracao(duracao, suporte=""):
    if pd.isna(duracao) or str(duracao).strip().lower() in ['s/d', '']:
        return "⏱️ Duração não informada"

    texto = str(duracao).lower()
    minutos_total = 0
    match_h = re.search(r"(\d+)\s*hora", texto)
    match_m = re.search(r"(\d+)\s*minuto", texto)
    match_s = re.search(r"(\d+)\s*segundo", texto)

    if match_h: minutos_total += int(match_h.group(1)) * 60
    if match_m: minutos_total += int(match_m.group(1))
    if match_s: minutos_total += int(match_s.group(1)) // 60

    suporte = suporte.lower()
    if "vídeo" in suporte or "video" in suporte:
        if minutos_total <= 5: return f"🎥 {duracao} (vídeo curto)"
        elif minutos_total <= 15: return f"🎬 {duracao} (vídeo médio)"
        else: return f"🧑‍🏫 {duracao} (vídeo longo)"
    elif "áudio" in suporte or "audio" in suporte:
        if minutos_total <= 5: return f"🎧 {duracao} (áudio curto)"
        elif minutos_total <= 20: return f"🎙️ {duracao} (áudio médio)"
        else: return f"🎚️ {duracao} (áudio longo)"
    elif "página" in texto:
        try:
            paginas = int(re.findall(r"\d+", texto)[0])
            if paginas <= 3: return f"📄 {duracao} (texto curto)"
            elif paginas <= 20: return f"📘 {duracao} (leitura média)"
            else: return f"📚 {duracao} (artigo longo)"
        except (IndexError, ValueError):
            return f"🖋️ {duracao}"
    return f"🖋️ {duracao}"

df_odas["Descricao_duracao"] = df_odas.apply(
    lambda row: interpretar_duracao(row.get("Duração/Tamanho"), row.get("Suporte", "")), axis=1
)

# === 4. TEXTO PARA EMBEDDING (ENRIQUECIDO) - VERSÃO ATUALIZADA ===
def preparar_texto_oda(row):
    """
    Cria um texto enriquecido para o embedding, incluindo metadados de IA no topo
    para contextualizar o modelo.
    """
    # Cria um cabeçalho com os metadados da IA. O .get() evita erros se a coluna não existir.
    rubrica = row.get('Rubrica_IA', 'Não classificado')
    confianca = row.get('Confiança_IA', 'Não avaliada')
    publico_alvo = row.get('Público-Alvo', 'Geral') # Adicionando o público-alvo, se existir

    # Monta uma frase que dá contexto explícito ao modelo de embedding
    header_inteligente = (
        f"Contexto deste material: "
        f"Destinado ao público '{publico_alvo}'. "
        f"Classificado na '{rubrica}' com confiança '{confianca}'."
    )

    partes = [
        header_inteligente, # O novo cabeçalho vai no início de tudo
        f"Título: {row.get('Título', '')}",
        f"Resumo: {row.get('Resumo', '')}",
        f"Resumo completo: {row.get('Resumo completo', '')}",
        f"Descrição: {row.get('Descrição', '')}",
        f"Conteúdo: {row.get('Conteúdo', '')}",
        f"Categoria: {row.get('Categoria', '')}",
        f"Temas: {row.get('Temas', '')}",
        f"Palavras-chave: {row.get('Palavras-chave - Termos do vocabulário controlado', '')}",
        f"Dimensões: {row.get('Dimensões', '')}",
        f"Temas em debate: {row.get('Temas em debate', '')}",
        f"Seção do Observatório: {row.get('Seção do Observatório', '')}",
        f"Instituições: {row.get('Instituições', '')}",
        f"Idioma: {row.get('Idiomas', '')}",
        f"Tipo: {row.get('Tipo', '')}",
        f"Tipo de Suporte: {row.get('Suporte', '')}",
        f"Duração: {row.get('Descricao_duracao', '')}"
    ]
    # Limpa partes vazias ou nulas
    return "\n".join([p for p in partes if pd.notnull(p) and str(p).strip() != "" and 'nan' not in str(p).lower()])

df_odas["texto_completo"] = df_odas.apply(preparar_texto_oda, axis=1)

# === 5. GERAR EMBEDDINGS ===
print("Iniciando a geração dos embeddings com o texto enriquecido...")
modelo = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
modelo = modelo.cuda()

embeddings = modelo.encode(
    df_odas["texto_completo"].tolist(),
    batch_size=16, # Aumentei um pouco o batch_size para performance, ajuste se der erro de memória
    show_progress_bar=True,
    convert_to_numpy=True
)
print("Embeddings gerados.")

# === 6. NORMALIZAR EMBEDDINGS ===
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# === 7. CRIAR FAISS INDEX ===
dimensao = embeddings.shape[1]
index = faiss.IndexFlatIP(dimensao)
index.add(embeddings.astype("float32"))

# === 8. SALVAR ARQUIVOS - VERSÃO ATUALIZADA ===
print("Salvando os novos arquivos de índice e metadados...")

# Garantir que todas as colunas estão presentes
colunas_existentes = df_odas.columns
colunas_para_salvar = [
    "Título", "Link fixo", "Fonte", "Resumo", "Resumo completo",
    "Descrição", "Conteúdo", "Categoria",
    "Suporte", "Tipo", "Dimensões", "Descricao_duracao",
    "Seção do Observatório", "Idiomas", "Instituições", "id",
    "Temas", "Palavras-chave - Termos do vocabulário controlado",
    "Temas em debate", "Rubrica_IA", "Confiança_IA", "Motivo_IA", "Rotulos_IA",
    #"Público-Alvo" # Adicionado para salvar, se existir
]

colunas_filtradas = [col for col in colunas_para_salvar if col in colunas_existentes]
metadados = df_odas[colunas_filtradas].reset_index(drop=True)

pasta_base = os.path.dirname(caminho_xlsx)

# ATUALIZANDO NOMES DOS ARQUIVOS PARA V2 (VERSÃO ENRIQUECIDA)
caminho_metadados = os.path.join(pasta_base, "metadados_odas_1606_v2.pkl")
caminho_index = os.path.join(pasta_base, "odas_index_1606_v2.faiss")

with open(caminho_metadados, "wb") as f:
    pickle.dump(metadados, f)

faiss.write_index(index, caminho_index)

# === 9. FIM ===
print("\n" + "="*50)
print(f"✅ Processo finalizado! Index criado com {index.ntotal} documentos.")
print(f"📂 Arquivos salvos em: {pasta_base}")
print(f"  - Metadados: {os.path.basename(caminho_metadados)}")
print(f"  - Índice:    {os.path.basename(caminho_index)}")
print("\n📋 Exemplo de texto enriquecido usado para o embedding:")
print(df_odas["texto_completo"].iloc[0])
print("="*50)

Iniciando a geração dos embeddings com o texto enriquecido...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Batches:   0%|          | 0/527 [00:00<?, ?it/s]

Embeddings gerados.
Salvando os novos arquivos de índice e metadados...

✅ Processo finalizado! Index criado com 8427 documentos.
📂 Arquivos salvos em: /content/drive/MyDrive/MORI
  - Metadados: metadados_odas_1606_v2.pkl
  - Índice:    odas_index_1606_v2.faiss

📋 Exemplo de texto enriquecido usado para o embedding:
Contexto deste material: Destinado ao público 'Geral'. Classificado na 'Sensibilização – Nível Avançar' com confiança '9'.
Título: Currículo Estadual de São Paulo – Etapa Ensino Médio
Resumo: O currículo escolar do Ensino Médio foi alvo de mudanças determinadas pela Lei 13.415/2017, que promoveu uma reorganização da Base Nacional Comum Curricular a partir de uma nova perspectiva composta pela Formação Geral Básica e por Itinerários Formativos. Neste material, foca-se nas aprendizagens dos estudantes, na formação inicial e continuada dos educadores, na produção de materiais de apoio, nas matrizes de avaliação e no estabelecimento de critérios para a oferta de infraestrutura 

In [ ]:
# === 8. SALVAR ===

# Garantir que todas as colunas estão presentes
colunas_existentes = df_odas.columns

colunas_para_salvar = [
    "Título", "Link fixo", "Fonte", "Resumo", "Resumo completo",
    "Descrição", "Conteúdo", "Categoria",
    "Suporte", "Tipo", "Dimensões", "Descricao_duracao", "texto_completo",
    "Seção do Observatório", "Idiomas", "Instituições", "id",
    "Temas", "Palavras-chave - Termos do vocabulário controlado",
    "Temas em debate", "Rubrica_IA", "Motivo_IA", "Rotulos_IA"
]

# Manter apenas colunas que realmente existem
colunas_filtradas = [col for col in colunas_para_salvar if col in colunas_existentes]

metadados = df_odas[colunas_filtradas].reset_index(drop=True)

# Caminho da pasta onde está o Excel original
pasta_base = os.path.dirname(caminho_xlsx)

# Definir caminhos de saída
caminho_metadados = os.path.join(pasta_base, "metadados_odas_1606.pkl")
caminho_index = os.path.join(pasta_base, "odas_index_1606.faiss")

# Salvar arquivos
with open(caminho_metadados, "wb") as f:
    pickle.dump(metadados, f)

faiss.write_index(index, caminho_index)

# === 9. FIM ===
print(f"✅ Index criado com {index.ntotal} documentos.")
print("📂 Arquivos salvos em:", pasta_base)
print("📋 Colunas incluídas nos metadados:", colunas_filtradas)
print(metadados.head(3))

✅ Index criado com 8427 documentos.
📂 Arquivos salvos em: /content/drive/MyDrive/MORI
📋 Colunas incluídas nos metadados: ['Título', 'Link fixo', 'Fonte', 'Resumo', 'Resumo completo', 'Suporte', 'Tipo', 'Dimensões', 'Descricao_duracao', 'texto_completo', 'Seção do Observatório', 'Idiomas', 'Instituições', 'id', 'Temas', 'Palavras-chave - Termos do vocabulário controlado', 'Rubrica_IA', 'Motivo_IA', 'Rotulos_IA']
                                              Título  \
0  Currículo Estadual de São Paulo – Etapa Ensino...   
1  Currículo do Espírito Santo – Texto introdutór...   
2  Currículo do Espírito Santo – Ciências Humanas...   

                                           Link fixo  \
0            a-curriculo-paulista-etapa-ensino-medio   
1  a-curriculo-do-espirito-santo-texto-introdutor...   
2  a-curriculo-do-espirito-santo-ciencias-humanas...   

                                               Fonte  \
0  https://efape.educacao.sp.gov.br/curriculopaul...   
1  https://drive.google